In [4]:
%pip install -q librosa soundfile torchaudio pandas numpy matplotlib tqdm pyyaml scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [5]:
import os
import json
import random
from pathlib import Path

import numpy as np 
import pandas as pd 

import matplotlib.pyplot as plt 

import tqdm

import soundfile as sf 

from IPython import display
from cProfile import label


In [6]:
PROJECT_ROOT = Path("C:/Users/Ayan/Desktop/Publication/Raag_Analysis")
RAW_DATA= PROJECT_ROOT / "data/raw"
PROCESSED = PROJECT_ROOT / "data/processed"
FEATURES = PROCESSED / "features/mel"
METADATA = PROCESSED / "metadata/unified/final_metadata.csv"
UPDATED_METADATA = PROCESSED / "metadata/unified/final_metadata_processed.csv"

In [7]:
assert RAW_DATA.exists()
 
print("Raw Data Found")

assert METADATA.exists()

print("Meta Data Found")

Raw Data Found
Meta Data Found


In [8]:
metadata = pd.read_csv(METADATA)

print(metadata.shape)

metadata.head()

metadata.columns

(1142, 16)


Index(['recording_id', 'dataset', 'audio_path', 'file_name', 'original_raag',
       'canonical_raag', 'duration_seconds', 'sample_rate', 'channels',
       'file_format', 'artist', 'album', 'instrument', 'tonic', 'split',
       'status'],
      dtype='str')

In [9]:
labels = sorted(metadata['canonical_raag'].dropna().astype(str).unique())

label_to_idx = {
    label: i 

    for i, label in enumerate(labels)
}

idx_to_label = {
    i:label

    for label,i in label_to_idx.items()
}



In [10]:


#DATASET SUMMARY
print ('=' *60)
print("DATASET SUMMARY")
print("=" * 60)

print(f"total records    :{len(metadata)}")
print(f"columns           : {len(metadata.columns)}")

print("columns: ")
for col in metadata.columns:
    print(f" - {col}")



# MISSING VALUES
print("\n" + "=" * 60)
print("MISSING VALUES")
print("=" * 60)

missing = metadata.isnull().sum()

missing = missing[missing > 0]

if len(missing) == 0:
    print("No missing values found.")

else:
    print(missing)


# DUPLICATE DETECTION
print("\n" + "=" * 60)
print("DUPLICATE RECORDS")

print("=" * 60)

duplicates = metadata.duplicated().sum()

print(f"Duplicate Rows: {duplicates}")


# AUDIO FILE EXISTS 
print("\n Checking Audio Paths... ")

missing_audio = []

for path in metadata["audio_path"]:
    audio_file = PROJECT_ROOT / path
    if not audio_file.exists():
        missing_audio.append(path)

print(f"Missing Audio Files: {len(missing_audio)}")

DATASET SUMMARY
total records    :1142
columns           : 16
columns: 
 - recording_id
 - dataset
 - audio_path
 - file_name
 - original_raag
 - canonical_raag
 - duration_seconds
 - sample_rate
 - channels
 - file_format
 - artist
 - album
 - instrument
 - tonic
 - split
 - status

MISSING VALUES
original_raag       15
canonical_raag      15
artist            1080
album             1034
instrument        1036
tonic             1034
split             1142
dtype: int64

DUPLICATE RECORDS
Duplicate Rows: 0

 Checking Audio Paths... 
Missing Audio Files: 0


In [11]:
metadata = metadata.dropna(subset=['canonical_raag']).copy()

metadata["canonical_raag"] = metadata["canonical_raag"].astype(str)

labels = sorted(metadata["canonical_raag"].unique())

# LABEL SUMMARY
print("=" * 60)
print("LABEL SUMMARY")
print("=" * 60)

print(f"Number of classes: {len(labels)}")

print("First 15 labels: ")
for label in labels[:15]:
    print(label)

# CLASS DISTRIBUTION
class_counts = metadata["canonical_raag"].value_counts()

print(class_counts.head(20))


LABEL SUMMARY
Number of classes: 99
First 15 labels: 
Abhogi
Adana
Ahir Bhairav
Alhaiya Bilaval
Asawari
Bageshree
Bahar
Bairagi
Barwa
Basant
Basant Mukhari
Basanti Kedar
Bhairav
Bhairav Bahar
Bhairavi
canonical_raag
Bhairavi                                32
Yaman                                   32
Bageshree                               30
Malkauns                                29
Bahar                                   27
Bhoopali                                27
Adana                                   26
Sarang                                  25
Basant                                  25
Deepak                                  25
Bhimpalasi                              24
Bihag                                   23
Desh                                    23
Khamaj                                  22
Miyan Ki Todi orTodi or Darbari Todi    22
Marwa                                   21
Kedar                                   21
Asawari                                 20
Bhatiyar  

In [12]:
label_to_idx = {
    label: idx 
    for idx, label in enumerate(labels)
}

idx_to_label = {
    idx: label

    for label, idx in label_to_idx.items()
}

import json

mapping_path = PROCESSED / "label_mapping.json"

with open(mapping_path, "w") as f:
    json.dump(label_to_idx, f, indent=4)

print("Saved:", mapping_path)

print("=" * 60)
print("LABEL ENCODER")
print("=" * 60)

print(f"Total Labels : {len(label_to_idx)}")

for i, (k, v) in enumerate(label_to_idx.items()):

    print(f"{k:25s} -> {v}")

    if i == 10:
        break

Saved: C:\Users\Ayan\Desktop\Publication\Raag_Analysis\data\processed\label_mapping.json
LABEL ENCODER
Total Labels : 99
Abhogi                    -> 0
Adana                     -> 1
Ahir Bhairav              -> 2
Alhaiya Bilaval           -> 3
Asawari                   -> 4
Bageshree                 -> 5
Bahar                     -> 6
Bairagi                   -> 7
Barwa                     -> 8
Basant                    -> 9
Basant Mukhari            -> 10


In [13]:

import random

sample = metadata.sample(5, random_state=42)

sample

print(metadata.columns.tolist())

['recording_id', 'dataset', 'audio_path', 'file_name', 'original_raag', 'canonical_raag', 'duration_seconds', 'sample_rate', 'channels', 'file_format', 'artist', 'album', 'instrument', 'tonic', 'split', 'status']


In [17]:
import librosa
from pathlib import Path
from tqdm.auto import tqdm

validation_results = []

def resolve_audio_path(relative_path: str) -> Path:
    relative = Path(relative_path)

    # Remove the leading data/raw if it exists
    if len(relative.parts) >= 2 and relative.parts[0] == "data" and relative.parts[1] == "raw":
        relative = Path(*relative.parts[2:])

    return RAW_DATA / relative

for _ , row in tqdm(metadata.iterrows(), total=len(metadata)):

    audio_file = resolve_audio_path(row["audio_path"])

    exists = audio_file.exists()

    readable = False
    actual_sr = None
    actual_duration = None

    if exists:

        try:

            y, sr = librosa.load(
                audio_file,
                sr=None,
                mono=False,
                duration=5
            )

            readable = True

            actual_sr = sr

            actual_duration = librosa.get_duration(path=audio_file)

        except Exception:

            pass

    validation_results.append({

        "recording_id": row["recording_id"],

        "exists": exists,

        "readable": readable,

        "metadata_sr": row["sample_rate"],

        "actual_sr": actual_sr,

        "metadata_duration": row["duration_seconds"],

        "actual_duration": actual_duration

    })

validation_df = pd.DataFrame(validation_results)

validation_df.head()


print("=" * 60)

print("AUDIO VALIDATION")

print("=" * 60)

print("Missing Files :",
      (~validation_df["exists"]).sum())

print("Unreadable Files :",
      (~validation_df["readable"]).sum())

print("Readable Files :",
      validation_df["readable"].sum())


print(RAW_DATA)

print(metadata["audio_path"].head(10).tolist())

sample = metadata.iloc[0]

resolved = resolve_audio_path(sample["audio_path"])

print("Original Audio Path")
print("audio_path:", sample["audio_path"])

print("Resolved Path: ")
print(resolved)

print("Exists: ")
print(resolved.exists())



100%|██████████| 1127/1127 [00:06<00:00, 161.39it/s]

AUDIO VALIDATION
Missing Files : 0
Unreadable Files : 0
Readable Files : 1127
C:\Users\Ayan\Desktop\Publication\Raag_Analysis\data\raw
['data/raw/saraga/audio/Anaahata by Milind Malshe/Raag Basanti Kedar/Raag Basanti Kedar.mp3.mp3', 'data/raw/saraga/audio/Anaahata by Milind Malshe/Raag Dagori_Deepki/Raag Dagori_Deepki.mp3.mp3', 'data/raw/saraga/audio/Anaahata by Milind Malshe/Raag Hindol Pancham/Raag Hindol Pancham.mp3.mp3', 'data/raw/saraga/audio/Anaahata by Milind Malshe/Raag Jait Kalyan/Raag Jait Kalyan.mp3.mp3', 'data/raw/saraga/audio/Anaahata by Milind Malshe/Raag Jogiya/Raag Jogiya.mp3.mp3', 'data/raw/saraga/audio/Anaahata by Milind Malshe/Raag Khat Todi/Raag Khat Todi.mp3.mp3', 'data/raw/saraga/audio/Anaahata by Milind Malshe/Raag Khokar/Raag Khokar.mp3.mp3', 'data/raw/saraga/audio/Anaahata by Milind Malshe/Raag Komal Rishav Aasavari/Raag Komal Rishav Aasavari.mp3.mp3', 'data/raw/saraga/audio/Anaahata by Milind Malshe/Raag Lalita Gauri/Raag Lalita Gauri.mp3.mp3', 'data/raw/sarag